# **Exploration**

## Questions
1. Import et choix de la source

2. Combien de pays distincts ? Combien de formats différents pour un même pays ?

3. Quelle proportion de lignes porte des caractères qui n'ont rien à faire dans un numéro de TVA ?

4. Combien de valeurs vides, et sous combien de formes différentes le vide se présente-t-il ?

5. Le pays déclaré dans le CRM correspond-il toujours au préfixe du numéro ?

6. Les canaux de saisie et les périodes de saisie se ressemblent-ils, ou certains sortent-ils du lot ?

---
## 1. Import et choix de la source

### Imports des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

### Chargement des données

In [6]:
data_dir = Path("../data")

df_csv = pd.read_csv(data_dir / "numeros_tva.csv")
df_xlsx = pd.read_excel(data_dir / "numeros_tva.xlsx")

### Inspection et comparaison des données

In [ ]:
# Dimensions
print(f"Shape CSV: {df_csv.shape}")
print(f"Shape Excel: {df_xlsx.shape}")

# Types des colonnes
print("\nTypes CSV:\n", df_csv.dtypes)
print("\nTypes Excel:\n", df_xlsx.dtypes)

# Égalité stricte
print("\nSont-ils strictement identiques ?", df_csv.equals(df_xlsx))

Shape CSV: (10000, 6)
Shape Excel: (10000, 6)

Types CSV:
 id                int64
raison_sociale      str
pays_declare        str
numero_tva          str
date_saisie         str
source_saisie       str
dtype: object

Types Excel:
 id                int64
raison_sociale      str
pays_declare        str
numero_tva          str
date_saisie         str
source_saisie       str
dtype: object

Sont-ils strictement identiques ? True


### Choix de la source de donnée

In [10]:
# On attribue le .csv comme variable pour le DataFrame
df = df_csv

### **Réponses**

- Les deux fichiers contiennent 10 000 lignes et 6 colonnes avec le meme typage.
<br>
<br>
<br>
- **Choix de la source : `numero_tva.csv`**  
Un xlsx a plus de chance d'avoir été modifié. et le .csv est plus performant (lecture plus rapide et consommation mémoire plus légère).

---
## 2. Combien de pays distincts ? Combien de formats différents pour un même pays ?

### Afficher les différents pays distincts

In [24]:
df["pays_declare"].value_counts()

pays_declare
FR    974
DK    961
BE    959
LU    956
SE    949
PT    947
NL    947
IT    947
PL    939
FI    902
ZZ    115
QQ    109
GB    104
UK    104
XX     87
Name: count, dtype: int64

### Compter le nombre de pays distincts

In [25]:
df["pays_declare"].nunique()

15

### Formats différents de TVA pour la France "FR"

In [26]:
filtre_fr = df[df["pays_declare"] == "FR"]

In [27]:
longueurs_fr = df[df["pays_declare"] == "FR"]["numero_tva"].str.len().value_counts()
print(longueurs_fr)

numero_tva
13.0    624
14.0    142
16.0     56
11.0     47
15.0     46
12.0     32
1.0      10
17.0      1
Name: count, dtype: int64


In [28]:
df[df["pays_declare"] == "FR"]["numero_tva"].str.len().nunique()

8

In [29]:
# Les 2 premiers caractères des numéros pour le pays déclaré FR
prefixes_fr = df[df["pays_declare"].str.upper() == "FR"]["numero_tva"].str[:2].nunique()
print(prefixes_fr)

44


### **Réponses**

- **Nombre de `pays_declare` : 15**  
  - Plusieurs ne sont pas de vrais pays (ZZ, QQ, XX)  
  - Plusieurs sont en dehors de l'Union Européenne (GB et UK)  
  - GB et UK sont le meme pays
- **Nombre de pays distincts : 11**
<br>
<br>
<br>
- Pour la France `pays_declare = "FR"` on observe
    - **8 longueurs de chaînes distinctes**  
    Numéros de 1 à 17 caractères, alors que le standard est de 13 caractères.
    - **44 préfixes différents**  
    Numéros démarrent avec 44 préfixes différents au lieu du seul FR attendu : présence de minuscules fr, d'espaces au début, de tirets, ou de chiffres directs sans code pays.

---
## 3. Quelle proportion de lignes porte des caractères qui n'ont rien à faire dans un numéro de TVA ?

### Compter les nombre de lignes avec probleme de caratére dans `numero_tva`

In [33]:
ligne_polluee = df["numero_tva"].str.contains(r"[^a-zA-Z0-9]", regex=True)

nb_lignes_ligne_polluees = ligne_polluee.sum()
proportion = ligne_polluee.mean() * 100

print(f"Nombre de lignes polluées : {nb_lignes_ligne_polluees}")
print(f"Proportion : {proportion:.2f} %")

Nombre de lignes polluées : 1813
Proportion : 18.13 %


### Afficher les caractères parasites

In [34]:
caracteres_parasites = df["numero_tva"].dropna().str.findall(r"[^a-zA-Z0-9]").explode().value_counts()
print(caracteres_parasites)

numero_tva
     2236
-     471
.     404
Name: count, dtype: int64


### **Réponses**

- **Nombre de lignes polluées : 1813**
<br>
<br>
- **Proportion : 18.13 %**
<br>
<br>
- Les caractéres parasites :
  - L'espace " " : 2236 occurrences
  - Le tiret "-" : 471 occurrences
  - Le point "." : 404 occurrences

---
## 4. Combien de valeurs vides, et sous combien de formes différentes le vide se présente-t-il ?

### Compter et afficher les valeurs vides différentes dans `numéro_tva`

In [49]:
# On charge les données brutes sans transformation de pandas
df_brut = pd.read_csv("../data/numeros_tva.csv", keep_default_na=False)

# On garde tout ce qui ne contient pas de chiffre
sans_chiffres = df_brut[~df_brut["numero_tva"].str.contains(r"\d", regex=True)]["numero_tva"]

# On renomme le vide et les espaces pour un affichage lisible
formes_lisibles = sans_chiffres.apply(lambda x: "[Chaîne vide]" if x == "" else ("[Espaces seuls]" if x.isspace() else x))

# On affiche chaque forme ligne par ligne
decompte = formes_lisibles.value_counts()
for forme, nb in decompte.items():
    print(f"Valeur {forme} : {nb}")

print("\nTotal des valeurs vides : ", len(sans_chiffres))

Valeur [Chaîne vide] : 60
Valeur [Espaces seuls] : 59
Valeur - : 55
Valeur N/A : 48
Valeur null : 38
Valeur NU.LL : 1

Total des valeurs vides :  261


### **Réponses**

- **Total des valeurs vides :  261**
    - Valeur [Chaîne vide] : 60 occurrences
    - Valeur [Espaces seuls] : 59 occurrences
    - Valeur - : 55 occurrences
    - Valeur N/A : 48 occurrences
    - Valeur null : 38 occurrences
    - Valeur NU.LL : 1 occurrence

---
## 5. Le pays déclaré dans le CRM correspond-il toujours au préfixe du numéro ?